# 第 2 课：广播机制（Broadcasting）

> 目标：理解不同形状的数组为什么能直接运算，以及怎么判断能不能运算。
> 广播是 NumPy 最核心、也是最容易"看着玄学"的机制，本课一次讲透。

## 一、什么是广播？

**广播 = 当两个数组形状不同时，NumPy 自动"扩展"它们，使形状一致后再做逐元素运算。**

注意：这个"扩展"是**概念上的**，NumPy 并不会真的复制数据（靠步长实现），所以既省内存又快。

其实你已经用过广播了——数组加一个数字：

In [1]:
import numpy as np

a = np.array([1, 2, 3])
print(a + 10)   # [11 12 13]
# 数字 10 被"广播"到数组的每个位置，相当于 [1,2,3] + [10,10,10]


[11 12 13]


## 二、广播三大规则（从右往左对齐维度）

对两个数组的 shape，**从最右边的维度开始向左对齐**，逐个维度检查：

1. **维度数量不同**时，在维度少的那个数组**左侧补 1**，直到维度数相同
2. 对齐后的两个维度，必须满足：**大小相等，或其中一个是 1**
3. 大小为 1 的维度，会被"复制"扩展成与另一个维度相同的大小

> 一句话：**右对齐，缺维补 1，遇 1 拉伸，其余必须相等。**
> 如果对齐后两个维度既不相等、也没有 1 → 报错 `could not be broadcast`。

## 三、例子 1：二维数组 + 一维数组（行广播）

In [ ]:
arr_2d = np.array([[1, 2, 3],
                   [4, 5, 6]])      # shape (2,3)
arr_1d = np.array([10, 20, 30])     # shape (3,)

# 第1步：(3,) 左侧补1 -> (1,3)
# 第2步：第0维 2 和 1，遇1拉伸 -> (2,3)，一维数组"复制"成两行
print(arr_2d + arr_1d)
# [[11 22 33]
#  [14 25 36]]


[[11 22 33]
 [14 25 36]]


## 四、例子 2：列向量 + 行向量（双向广播，外积）

`(3,1)` 和 `(1,3)`：
- 第0维：3 和 1 → 1 拉伸成 3
- 第1维：1 和 3 → 1 拉伸成 3
- 结果 `(3,3)`

In [ ]:
col = np.array([[10], [20], [30]])   # shape (3,1)
row = np.array([1, 2, 3])            # shape (3,)，补1成 (1,3)

print(col + row)
# [[11 12 13]
#  [21 22 23]
#  [31 32 33]]

# 这就是数学上的"外积"思想：列向量 × 行向量
print(col * row)
# [[10 20 30]
#  [20 40 60]
#  [30 60 90]]


[[11 12 13]
 [21 22 23]
 [31 32 33]]
[[10 20 30]
 [20 40 60]
 [30 60 90]]


## 五、例子 3：标量与数组（最简单的广播）

In [4]:
a = np.ones((2, 3))
print(a * 5)      # 标量 5 广播到全部位置
print(a.shape)    # (2,3)，运算不改变原数组形状


[[5. 5. 5.]
 [5. 5. 5.]]
(2, 3)


## 六、不能广播的情况（看懂报错）

In [5]:
# (2,3) 和 (2,)：右对齐 -> 第1维 3 和 2，既不相等也没有1 -> 报错
a = np.ones((2, 3))
b = np.array([1, 2])     # shape (2,)
try:
    print(a + b)
except ValueError as e:
    print("报错：", e)


报错： operands could not be broadcast together with shapes (2,3) (2,) 


**修正方法**：把 b 显式变成列向量 `(2,1)`：

In [6]:
b_col = b[:, np.newaxis]   # shape (2,1)
print(a + b_col)
# [[2. 2. 2.]
#  [3. 3. 3.]]


[[2. 2. 2.]
 [3. 3. 3.]]


## 七、广播的实际用途（重点，建立工程感）

### 用途 1：数据标准化（每个数减去列均值）
机器学习预处理最常见操作：让每列数据减去自己的平均值。

In [7]:
data = np.array([[10.0, 100.0],
                 [20.0, 200.0],
                 [30.0, 300.0]])
col_mean = data.mean(axis=0)     # [20. 200]，shape (2,)
print(data - col_mean)           # (3,2)-(2,) 行广播，每行都减列均值
# [[-10. -100.]
#  [  0.    0.]
#  [ 10.  100.]]


[[ -10. -100.]
 [   0.    0.]
 [  10.  100.]]


### 用途 2：批量加偏置 / 批量缩放
给一个矩阵的每一行统一加上同一个向量（神经网络里加偏置 b 就是这个操作）。

### 用途 3：距离矩阵、网格计算（第 4 课实战会做）

## 八、安全建议（避免广播悄悄出错）

广播很强大，但形状"碰巧能广播"时可能得到你不想要的结果。建议：
1. **运算前先打印两个数组的 shape**，心算广播结果
2. 想要列方向的运算，**显式 `[:, np.newaxis]`**，比依赖自动广播更清晰
3. 运算后检查结果 shape 是否符合预期

---

## 课后练习

**练习1**：判断下列 shape 组合能否广播，结果 shape 是什么？（先写答案再验证）
- `(3,4)` 和 `(4,)`
- `(3,4)` 和 `(3,1)`
- `(3,1,4)` 和 `(2,4)`
- `(3,4)` 和 `(3,)`

**练习2**：生成一个 4 行 3 列的数组，让**每一行**都减去该行第一个元素的值（提示：需要列向量形状的"每行首元素"）。

**练习3**：用列向量 `[1,2,3,4]`（shape (4,1)）和行向量 `[10,20]`（shape (2,)）通过广播相加，写出结果。

---
## 参考答案

### 练习1
- `(3,4)` + `(4,)`：(4,)补1成(1,4) → 结果 **(3,4)**
- `(3,4)` + `(3,1)`：第1维 4 和 1 拉伸 → 结果 **(3,4)**
- `(3,1,4)` + `(2,4)`：补1对齐后 (3,1,4) 和 (1,2,4)，两个1分别拉伸 → 结果 **(3,2,4)**
- `(3,4)` + `(3,)`：右对齐 4 和 3 冲突 → **不能广播，报错**

### 练习2

In [8]:
a = np.arange(12).reshape(4, 3)
first_col = a[:, 0, np.newaxis]   # shape (4,1)，每行首元素，保持列向量
print(a - first_col)              # (4,3)-(4,1) 广播


[[0 1 2]
 [0 1 2]
 [0 1 2]
 [0 1 2]]


### 练习3

In [9]:
col = np.array([[1],[2],[3],[4]])   # (4,1)
row = np.array([10, 20])            # (2,) -> (1,2)
print(col + row)
# [[11 21]
#  [12 22]
#  [13 23]
#  [14 24]]


[[11 21]
 [12 22]
 [13 23]
 [14 24]]
